# TP8 : atelier incidents en binômes

| Repère | Valeur |
|---|---|
| Moment | Jour 2, 15:35-16:10, après la démo du TP (15:30-15:35) |
| Compétences | C9 Listeners, TLS et SASL ; C13 Diagnostic ; C16 Méthode de diagnostic d'incident |
| Slides de rappel | S70 (méthode), S71 (arbre de décision et messages d'erreur), S72 (boîte à outils) |
| Fiche à remplir | `fiche-diagnostic.md`, une copie par ticket (clic droit sur le fichier, *Duplicate*) |
| Version Markdown | `enonce.md`, mêmes tickets en ligne de commande seule |

**Mode d'emploi du notebook**

- Exécutez d'abord la cellule d'outils, puis l'étape 6.1. Après un redémarrage du noyau, relancez la cellule d'outils et la cellule de 6.1.
- Les sections 6.2 à 6.4 sont les trois tickets. Traitez-les dans l'ordre de vos tours (tableau de 6.1), pas dans l'ordre du notebook.
- Dans chaque ticket : une **cellule de reproduction** fournie, une **zone `# TODO` de diagnostic** (ajoutez autant de cellules que nécessaire, bouton `+`), une **zone `# TODO` de correction**, puis une **vérification**, qui contrôle l'objectif de résolution du ticket.
- **Règle du jeu** : ne lisez pas `scripts/incident.sh` et n'utilisez pas ses commandes `status` : elles donnent la réponse. Le diagnostic se fait avec la boîte à outils ci-dessous, comme en production.
- Les messages qui commencent par `%3|` ou `%4|` viennent de la bibliothèque cliente (librdkafka) : ce sont des erreurs et avertissements du client Python. Ils apparaissent sous les cellules qui produisent ou consomment des messages, comme les cellules de reproduction. Lisez-les : ce sont des indices.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import subprocess
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
PROMETHEUS = "http://prometheus:9090"
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Deux fonctions d'aide, écrites ici : elles ne font pas partie de confluent-kafka
def wait_for(condition, timeout=60, step=2, message="condition"):
    """Rappelle condition() jusqu'à obtenir une valeur vraie, et la renvoie."""
    start = time.time()
    while True:
        try:
            value = condition()
        except Exception:
            value = None
        if value:
            return value
        if time.time() - start >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(step)

def table(lines, headers):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in line) + " |" for line in lines]
    display(Markdown("\n".join(md)))

In [ ]:
import os
import time
from collections import Counter

import requests
from confluent_kafka import OFFSET_BEGINNING, Consumer, KafkaError, KafkaException, Producer, TopicPartition
from confluent_kafka.admin import AlterConfigOpType, ConfigEntry, ConfigResource, ConfigSource, OffsetSpec

# Boîte à outils du diagnostic (section 8.3 de la version Markdown)


def promql(query, label=None):
    """Requête PromQL instantanée : {valeur de l'étiquette: valeur}."""
    results = requests.get(PROMETHEUS + "/api/v1/query", params={"query": query}, timeout=10).json()["data"]["result"]
    return {r["metric"].get(label, "") if label else "": float(r["value"][1]) for r in results}


def alerts():
    """Alertes actives de Prometheus : ["nom:etat", ...], avec (xN) quand plusieurs nœuds sont concernés."""
    data = requests.get(PROMETHEUS + "/api/v1/alerts", timeout=10).json()["data"]["alerts"]
    account = Counter("%s:%s" % (a["labels"]["alertname"], a["state"]) for a in data)
    return sorted(key + (" (x%d)" % n if n > 1 else "") for key, n in account.items())


def under_min_isr():
    """À écrire à l'étape 6.1."""
    return None


def with_retry(func, attempts=8):
    """Appelle func() ; en cas d'erreur Kafka, nouvel essai (admin crée un client neuf à chaque appel).

    describe_cluster() et describe_configs() partent vers le broker que les métadonnées désignent comme
    contrôleur, tiré au hasard en KRaft à chaque réponse : s'il est injoignable, la requête échoue au bout
    du délai, et l'essai suivant a de bonnes chances de viser un autre broker.
    """
    admin = AdminClient(CONF)
    for attempt in range(attempts):
        try:
            return func()
        except KafkaException as e:
            if attempt == attempts - 1:
                raise
            print("[%s : %s, nouvel essai avec un client neuf]" % (func.__name__, e.args[0].name()))


def summary(show=True):
    """État général, du quorum aux alertes : la référence « tout va bien » de 6.1 sert de comparaison."""
    admin = AdminClient(CONF)
    status = !docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --status 2>&1
    status = status.n
    metadata = admin.list_topics(timeout=10)
    partitions = [p for t in metadata.topics.values() for p in t.partitions.values()]
    result = {
        "quorum": "leader %s, lag max %s" % (status.split("LeaderId:")[1].split()[0],
                                             status.split("MaxFollowerLag:")[1].split()[0]),
        # Noms annoncés, lus dans les métadonnées : elles viennent de n'importe quel broker joignable
        "brokers": {b.id: b.host for b in sorted(metadata.brokers.values(), key=lambda b: b.id)},
        "urp": sum(1 for p in partitions if len(p.isrs) < len(p.replicas)),
        "sous_min_isr": with_retry(under_min_isr),
        "indisponibles": sum(1 for p in partitions if p.leader == -1),
        "alertes": alerts(),
    }
    if show:
        print("== %s (heure UTC du conteneur)" % time.strftime("%H:%M:%S"))
        for key, value in result.items():
            print("%-14s %s" % (key, value))
    return result


def describe(topic):
    """Leader, répliques et ISR de chaque partition d'un topic."""
    admin = AdminClient(CONF)
    partitions = admin.list_topics(topic, timeout=10).topics[topic].partitions
    table([(p.id, p.leader, p.replicas, p.isrs) for p in sorted(partitions.values(), key=lambda p: p.id)],
          ["Partition", "Leader", "Répliques", "ISR"])


def topic_config(topic):
    """Réglages propres à un topic (source DYNAMIC_TOPIC_CONFIG) : {name: value}."""
    admin = AdminClient(CONF)
    future = list(admin.describe_configs([ConfigResource("topic", topic)]).values())[0]
    return {name: e.value for name, e in future.result().items() if e.source == ConfigSource.DYNAMIC_TOPIC_CONFIG.value}


def logs(node, pattern, since="3m", lines=10):
    """Dernières lignes des traces d'un nœud qui correspondent au motif (expression régulière étendue)."""
    output = !docker compose logs kafka-{node} --since {since} --no-log-prefix 2>&1 | grep -E '{pattern}' | tail -{lines} | cut -c1-250
    return output.n


def sizes_per_broker():
    """Données stockées par nœud, en Mo (métrique kafka_log_log_size collectée par Prometheus)."""
    return dict(sorted((n, round(v / 1048576)) for n, v in
                       promql("sum by (instance) (kafka_log_log_size)", "instance").items()))


def sizes_per_topic():
    """Données stockées par topic, toutes répliques confondues, en Mo, du plus gros au plus petit."""
    return dict(sorted(((t, round(v / 1048576, 1)) for t, v in
                        promql("sum by (topic) (kafka_log_log_size)", "topic").items()), key=lambda x: -x[1]))


def watch(count, step=15, until=None):
    """Relève, toutes les step secondes, la taille par nœud et les alertes ; s'arrête dès que until(alerts) est vrai."""
    start = time.time()
    for _ in range(count):
        time.sleep(step)
        active = alerts()
        print("t+%3ds  %s  alertes: %s" % (time.time() - start, "  ".join("%s=%dMo" % kv for kv in
                                                                          sizes_per_broker().items()), " ".join(active)))
        if until and until(active):
            break


def read_all(topic, delay=10):
    """Toutes les clés et valeurs d'un topic, depuis le début (lecture par assign, sans groupe)."""
    admin = AdminClient(CONF)
    partitions = admin.list_topics(topic, timeout=10).topics[topic].partitions
    consumer = Consumer({"bootstrap.servers": BS, "group.id": "tp8-lecture", "enable.auto.commit": False,
                         "enable.partition.eof": True, "log_level": 4})
    consumer.assign([TopicPartition(topic, p, OFFSET_BEGINNING) for p in partitions])
    messages, finished, deadline = [], set(), time.time() + delay
    while len(finished) < len(partitions) and time.time() < deadline:
        msg = consumer.poll(1)
        if msg is None:
            continue
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                finished.add(msg.partition())
            continue
        messages.append(((msg.key() or b"").decode(), msg.value().decode()))
    consumer.close()
    return messages


def message_count(topic):
    """Nombre de messages encore présents dans un topic : fin moins début, sur toutes ses partitions."""
    admin = AdminClient(CONF)
    partitions = admin.list_topics(topic, timeout=10).topics[topic].partitions

    def offsets(spec):
        offset_requests = {TopicPartition(topic, p): spec for p in partitions}
        return {tp.partition: f.result().offset for tp, f in admin.list_offsets(offset_requests).items()}

    start, end = offsets(OffsetSpec.earliest()), offsets(OffsetSpec.latest())
    return sum(end[p] - start[p] for p in partitions)


print("Prometheus joignable :", requests.get(PROMETHEUS + "/-/ready", timeout=5).status_code == 200)

## 1. Objectifs

À la fin du TP, vous savez :

- qualifier un incident à partir d'un ticket et d'un message d'erreur ;
- observer le cluster du général au particulier, avant d'agir ;
- formuler une hypothèse et la vérifier par une commande ;
- corriger par l'action la moins risquée, puis prouver le retour à la normale ;
- rédiger une fiche d'incident exploitable par une autre équipe.

## 2. Prérequis

- Lab en mode `extended`, sain : `scripts/validate.sh` sans échec. Les incidents utilisent le générateur de charge et Prometheus.
- TP5 et TP6 terminés : listener SECURE, comptes SCRAM, requêtes Prometheus et alertes.
- Une copie de `fiche-diagnostic.md` par ticket.
- Notions : slides S70 à S72, et S71 en particulier pour les messages d'erreur.

## 3. Architecture utilisée

```
  Binôme : stagiaire A (lab A) et stagiaire B (lab B)

  Tour n :  B injecte un incident sur le lab A  --->  A diagnostique et corrige sur son lab
            A injecte un incident sur le lab B  --->  B diagnostique et corrige sur son lab
            (en parallèle, puis échange de fiches)

  Lab : kafka-1..3 (broker et contrôleur), listeners INTERNAL :19092 et SECURE :19094 (SASL_SSL),
        loadgen 100 messages/s, Prometheus :9090 et ses alertes, Grafana :3000
```

L'injecteur connaît l'incident et joue l'équipe qui a ouvert le ticket. Il répond aux questions sur le contexte, jamais sur la cause.

| Lab | Production |
|---|---|
| L'incident est injecté par un script, et réparé par le même script en fin de tour | L'incident n'est pas connu ; la correction passe par la gestion de configuration et le processus de changement |
| Seuil disque de l'alerte `KafkaBrokerLogSizeHigh` : 250 Mo par nœud | Seuil en pourcentage du disque, mesuré par l'exporteur du système (node_exporter) |
| Vérification de la rétention toutes les 30 s | 5 minutes par défaut (`log.retention.check.interval.ms`) |

## 4. Contexte métier

Fin d'après-midi chez NovaShop. Trois tickets arrivent à l'équipe d'exploitation Kafka, dont vous assurez l'astreinte. Chacun doit être traité avec méthode : une correction hasardeuse sur le cluster de production coûterait plus cher que l'incident lui-même.

## 5. Énoncé

Chaque stagiaire traite les trois tickets, sur son propre lab, dans l'ordre fixé par le tableau de 6.1. Pour chacun : diagnostic, correction, validation et fiche remplie.

| Ticket | Équipe | Texte du ticket | Objectif de résolution |
|---|---|---|---|
| **Logistique** | Logistique | « Notre nouveau service `shipping-service` doit publier ses événements d'expédition dans le topic `shipments`, sur le listener sécurisé. On nous a créé le compte `shipping` et on a livré notre fichier de configuration. L'envoi échoue. Notre script de test : `/tmp/incident4/send-shipment.sh` dans `tools`. » | Le script se termine sans erreur et l'événement est lu dans `shipments`. |
| **Paiements** | Paiements | « Depuis une intervention sur `kafka-3`, une partie des paiements n'est plus enregistrée : certains envois expirent, d'autres passent. Les alertes de réplication se sont allumées. » | Tous les envois passent ; URP et partitions sous min ISR à 0 ; alertes levées. |
| **Stockage** | Supervision | « L'alerte `KafkaBrokerLogSizeHigh` s'est déclenchée sur les 3 nœuds. L'équipe Data a lancé une collecte de `clickstream` cet après-midi. Sa politique : garder au plus 1 heure et 5 Mo par partition. » | Volume revenu sous le seuil, limite en place contre la récidive, sans supprimer le topic ; alerte levée. |

## 6. Étapes

### 6.1 Préparer le binôme et le lab

Fixez l'ordre des tickets :

| Tour | Lab de A (injecté par B) | Lab de B (injecté par A) |
|---|---|---|
| 1 | Paiements | Logistique |
| 2 | Stockage | Paiements |
| 3 | Logistique | Stockage |

Correspondance, **réservée à l'injecteur** : Logistique = 4, Paiements = 5, Stockage = 6. Injection, dans un terminal du poste de votre partenaire, dossier `lab/` : `scripts/incident.sh N inject`.

La fonction `summary()` de la boîte à outils donne l'état général. Il lui manque un indicateur : le nombre de partitions **sous min ISR**, celles qui refusent les écritures `acks=all`.

**Exemple en ligne de commande** : les trois compteurs de partitions dégradées.

In [ ]:
!docker exec kafka-tools bash -c 'for option in under-replicated-partitions under-min-isr-partitions unavailable-partitions; do echo "$$option : $$(kafka-topics.sh --bootstrap-server $BS --describe --$$option | grep -c Partition)"; done'

In [ ]:
# TODO 6.1 : compléter la boîte à outils, puis enregistrer le bilan de référence
# 1. Écrivez under_min_isr() : nombre de partitions dont l'ISR compte moins de répliques que le min.insync.replicas
#    de leur topic. Lisez min.insync.replicas de tous les topics en un seul appel :
#    admin.describe_configs([ConfigResource("topic", t) for t in topics], request_timeout=5)
#    renvoie {resource: future} ;
#    resource.name est le nom du topic, future.result()["min.insync.replicas"].value sa valeur (texte).
# 2. référence = summary() : c'est votre état « tout va bien », à comparer à chaque fin de tour.
def under_min_isr():
    return None


reference = {}

In [ ]:
# Vérification 6.1
assert reference.get("sous_min_isr") == 0, "À revoir : sous_min_isr() écrite : aucune partition sous min ISR"
print("OK  sous_min_isr() écrite : aucune partition sous min ISR")
assert reference.get("brokers") == {1: "kafka-1", 2: "kafka-2", 3: "kafka-3"}, "À revoir : 3 brokers, annoncés sous leur nom"
print("OK  3 brokers, annoncés sous leur nom")
assert reference.get("urp") == 0 and reference.get("indisponibles") == 0, "À revoir : aucune partition dégradée"
print("OK  aucune partition dégradée")
assert reference.get("alertes") == [], "À revoir : aucune alerte active : signalez au formateur toute alerte déjà présente"
print("OK  aucune alerte active : signalez au formateur toute alerte déjà présente")

**Résultat attendu** : un leader de quorum, les brokers `kafka-1`, `kafka-2` et `kafka-3`, les trois compteurs à 0 et aucune alerte. Juste après le TP7, une alerte `pending` (`KafkaFencedBrokers`, `KafkaUnderMinIsrPartitions`) peut subsister une trentaine de secondes après l'arrêt de `kafka-4` : attendez, puis relancez `summary()`. Si une alerte reste active, signalez-le au formateur avant de commencer.

### 6.2 Ticket Logistique

> « Notre nouveau service `shipping-service` doit publier ses événements d'expédition dans le topic `shipments`, sur le listener sécurisé. On nous a créé le compte `shipping` et on a livré notre fichier de configuration. L'envoi échoue. Notre script de test : `/tmp/incident4/send-shipment.sh` dans `tools`. »

**Reproduction** : le script de test de l'équipe, lancé dans `tools`. La fonction retire de sa sortie les lignes `at ...` des piles d'appels Java, puis compte les lignes `ERROR`.

Le code retour vaut 0 même quand l'envoi échoue : `kafka-console-producer.sh` ne signale pas un échec d'envoi par son code retour. Ce sont les lignes `ERROR` qui comptent.

In [ ]:
def reproduce_logistics():
    command = "/tmp/incident4/send-shipment.sh 2>&1 | grep -v '^[[:space:]]*at '; exit ${PIPESTATUS[0]}"
    result = subprocess.run(["docker", "exec", "kafka-tools", "bash", "-c", command], capture_output=True, text=True)
    output = result.stdout.rstrip("\n")
    print("code retour : %d, lignes ERROR : %d" % (result.returncode, output.count(" ERROR ")))
    return output

In [ ]:
# TODO 6.2 a : diagnostic du ticket Logistique
# Reproduisez, observez du général au particulier, puis vérifiez chaque hypothèse par une commande.
# Ajoutez une cellule par hypothèse. Notez dans la fiche chaque commande et son constat.
# Boîte à outils : summary(), describe(topic), topic_config(topic), logs(node, pattern), alerts(), promql(...),
# des cellules `!docker exec kafka-tools ...` et `!docker compose ...` ; commandes de la section 8.3 de enonce.md.
reproduce_logistics()

In [ ]:
# TODO 6.2 b : correction du ticket Logistique
# Appliquez la correction la moins risquée (commande `!docker exec kafka-tools ...`, `!docker compose ...` ou appel `admin`), puis relancez reproduce_logistics().

In [ ]:
# Vérification 6.2
attempt = reproduce_logistics()
assert "ERROR" not in attempt, "À revoir : le script de test se termine sans erreur"
print("OK  le script de test se termine sans erreur")
shipments = read_all("shipments")
assert any("colissimo" in value for _, value in shipments), "À revoir : événement d'expédition lu dans shipments (%d message(s))" % len(shipments)
print("OK  événement d'expédition lu dans shipments (%d message(s))" % len(shipments))
print(shipments[-1])

**Résultat attendu** :

- le script se termine sans aucune ligne `ERROR`, avec le code retour 0 ;
- la lecture de `shipments` depuis le début affiche l'événement `SHIP-<horodatage>` avec `"carrier":"colissimo"` ;
- les traces des brokers ne montrent plus de nouvel échec d'authentification pour ce client.

**Fin du tour** : l'injecteur lance `scripts/incident.sh 4 repair` dans un terminal de votre poste. Il réapplique l'état de référence, même si votre correction est déjà la bonne, et envoie un événement de plus. Comparez ensuite le bilan à la référence de 6.1.

In [ ]:
summary()

### 6.3 Ticket Paiements

> « Depuis une intervention sur `kafka-3`, une partie des paiements n'est plus enregistrée : certains envois expirent, d'autres passent. Les alertes de réplication se sont allumées. »

Attendez 30 s après le `[ OK ]` de l'injection avant la première observation.

**Reproduction** : 6 paiements, un par clé, chacun avec 20 s pour être confirmé. Le partitionneur `murmur2_random` place chaque clé dans la même partition que le client Java de la version Markdown.

```bash
# [tools] equivalent en ligne de commande
for k in PAY-9001 PAY-9002 PAY-9003 PAY-9004 PAY-9005 PAY-9006; do echo "$k|{\"amount\":42}"; done \
  | kafka-console-producer.sh --bootstrap-server $BS --topic payments \
      --reader-property parse.key=true --reader-property 'key.separator=|' \
      --command-property delivery.timeout.ms=20000 --command-property request.timeout.ms=10000
```

In [ ]:
def reproduce_payments():
    results = {}

    def report(err, msg):
        results[msg.key().decode()] = ("payments-%d" % msg.partition(), err.name() if err else "OK")

    producer = Producer({"bootstrap.servers": BS, "partitioner": "murmur2_random", "log_level": 4,
                         "delivery.timeout.ms": 20000, "request.timeout.ms": 10000})
    for number in range(9001, 9007):
        producer.produce("payments", key="PAY-%d" % number, value='{"amount":42}', on_delivery=report)
    producer.flush(30)
    table([(key,) + result for key, result in sorted(results.items())], ["Clé", "Partition", "Résultat"])
    return results

In [ ]:
# TODO 6.3 a : diagnostic du ticket Paiements
# Quels envois passent, lesquels échouent, sur quelles partitions ? Puis observez du général au particulier.
# Ajoutez une cellule par hypothèse. Notez dans la fiche chaque commande et son constat.
reproduce_payments()

In [ ]:
# TODO 6.3 b : correction du ticket Paiements
# Appliquez la correction la moins risquée, sur un seul nœud, puis attendez son retour (sain, URP à 0).
# Les alertes de réplication se lèvent 1 à 2 minutes après la correction.

In [ ]:
# Vérification 6.3
results = reproduce_payments()
assert len(results) == 6 and all(r == "OK" for _, r in results.values()), "À revoir : les 6 paiements passent"
print("OK  les 6 paiements passent")
state = summary()
assert state["brokers"] == {1: "kafka-1", 2: "kafka-2", 3: "kafka-3"}, "À revoir : brokers annoncés sous leur nom"
print("OK  brokers annoncés sous leur nom")
assert state["urp"] == 0 and state["sous_min_isr"] == 0 and state["indisponibles"] == 0, "À revoir : URP, sous min ISR et indisponibles à 0"
print("OK  URP, sous min ISR et indisponibles à 0")
assert not [a for a in state["alertes"] if a.startswith(("KafkaUnderReplicated", "KafkaUnderMinIsr"))], "À revoir : alertes de réplication levées (sinon, relancez cette cellule dans 1 minute)"
print("OK  alertes de réplication levées (sinon, relancez cette cellule dans 1 minute)")

**Résultat attendu** :

- les 6 envois passent ;
- `summary()` : les brokers sont annoncés sous les noms `kafka-1`, `kafka-2`, `kafka-3` ; URP, sous min ISR et indisponibles à 0 ;
- les alertes de réplication sont levées dans les 1 à 2 minutes qui suivent la correction.

**Fin du tour** : l'injecteur lance `scripts/incident.sh 5 repair` dans un terminal de votre poste. Il redémarre `kafka-3` une seconde fois, environ 15 s. Attendez que l'URP revienne à 0, puis comparez le bilan à la référence de 6.1.

In [ ]:
summary()

### 6.4 Ticket Stockage

> « L'alerte `KafkaBrokerLogSizeHigh` s'est déclenchée sur les 3 nœuds. L'équipe Data a lancé une collecte de `clickstream` cet après-midi. Sa politique : garder au plus 1 heure et 5 Mo par partition. »

Attendez l'alerte : 1 min 30 à 2 min après l'injection, selon le volume déjà présent sur votre lab. La fonction `watch()` de la boîte à outils relève la taille par nœud et les alertes toutes les 15 s.

**Reproduction** : les alertes, et la taille des données par nœud (panneau « Donnees stockees par broker » de Grafana).

In [ ]:
def reproduce_storage():
    print("alertes :", alerts())
    print("données par nœud (Mo) :", sizes_per_broker())

In [ ]:
# TODO 6.4 a : diagnostic du ticket Stockage
# Quel topic grossit ? Pourquoi ? Ajoutez une cellule par hypothèse. Notez dans la fiche chaque commande et son constat.
reproduce_storage()

In [ ]:
# TODO 6.4 b : correction du ticket Stockage
# Appliquez une correction conforme à la politique de l'équipe Data, sans supprimer le topic ni ses fichiers.
# Puis suivez le retour sous le seuil : watch(8, until=lambda active: not any(...)).

In [ ]:
# Vérification 6.4
settings = topic_config("clickstream")
assert 0 < int(settings.get("retention.ms", -1)) <= 3600000 and 0 < int(settings.get("retention.bytes", -1)) <= 5242880, "À revoir : clickstream limité à 1 h et 5 Mo par partition"
print("OK  clickstream limité à 1 h et 5 Mo par partition")
sizes = sizes_per_broker()
assert sizes and max(sizes.values()) < 250, "À revoir : moins de 250 Mo par nœud : %s" % sizes
print("OK  moins de 250 Mo par nœud : %s" % sizes)
assert not [a for a in alerts() if a.startswith("KafkaBrokerLogSizeHigh")], "À revoir : alerte KafkaBrokerLogSizeHigh levée (sinon, relancez cette cellule dans 30 s)"
print("OK  alerte KafkaBrokerLogSizeHigh levée (sinon, relancez cette cellule dans 30 s)")
assert message_count("clickstream") > 0, "À revoir : topic présent, avec %d messages récents" % message_count("clickstream")
print("OK  topic présent, avec %d messages récents" % message_count("clickstream"))

**Résultat attendu** :

- la configuration de `clickstream` porte une limite de durée et de taille conforme à la politique de l'équipe Data ;
- moins de 250 Mo par nœud ;
- `KafkaBrokerLogSizeHigh` absente des alertes, environ 1 minute après la correction ;
- le topic existe toujours, avec ses données récentes.

**Fin du tour** : l'injecteur lance `scripts/incident.sh 6 repair` dans un terminal de votre poste : il réapplique la même rétention. Comparez ensuite le bilan à la référence de 6.1.

In [ ]:
summary()

## Bilan de l'exercice

À comparer au bilan de référence de 6.1. Les trois fiches remplies, relues et signées par l'injecteur, complètent ce bilan.

In [ ]:
final = summary(show=False)
backup_incident5 = os.path.exists(os.path.join(LAB_DIR, "config/kafka/kafka-3/server.properties.before-incident5"))
table([(key, final[key], reference.get(key, "à compléter en 6.1")) for key in ("brokers", "urp", "sous_min_isr",
                                                                                  "indisponibles", "alertes")]
        + [("sauvegarde de l'incident 5 présente", backup_incident5, False)],
       ["Contrôle", "Fin du TP", "Référence (6.1)"])

## 7. Questions de réflexion

1. Ticket Paiements : pourquoi certains envois passent-ils alors que d'autres expirent ?
2. Ticket Paiements : `kafka-3` est `healthy` pour Docker et `unfenced` pour le contrôleur. Pourquoi est-il pourtant inutilisable pour les clients ?
3. Ticket Paiements : pourquoi l'alerte `KafkaNodeDown` ne s'est-elle pas déclenchée ?
4. Ticket Logistique : pourquoi le client a-t-il rencontré deux erreurs successives ? Que disaient les traces du broker dans chaque cas ?
5. Ticket Logistique : fallait-il corriger le client ou le broker ? Pourquoi ?
6. Ticket Stockage : pourquoi une limite de durée d'une heure, seule, n'aurait-elle rien libéré ?
7. Ticket Stockage : pourquoi l'espace n'est-il pas libéré à l'instant même de la correction ?
8. Pour chaque ticket, quelle mesure de prévention proposez-vous ?